# Análisis Versión 4: Revisión y evaluación de modelos

**Proyecto:** Estimación de producción y clasificación de campaña en el cultivo de arroz en Bolivia  
**Área:** Aprendizaje automático supervisado  

Este informe reúne los resultados de la versión V4 del proyecto. En esta etapa se revisan con cuidado las conclusiones de los análisis anteriores, se profundiza en las características reales de la base de datos y se evalúan los modelos bajo condiciones parejas y realistas. El documento está escrito para que sea comprensible tanto para personas con formación técnica como para quienes buscan entender qué nos dicen los datos sin enredarse en tecnicismos innecesarios.

---

## 1. Revisión y evaluación

### 1.1 Motivación y limitaciones detectadas en los primeros análisis
En las primeras pruebas del proyecto (versiones V1 a V3) se observaron números llamativos, como modelos con coeficientes de determinación ($R^2$) cercanos a 0.77 en regresión o clasificaciones con más del 79% de exactitud. Sin embargo, al examinar en detalle los experimentos, se encontraron cinco limitaciones metodológicas que obligaron a hacer una revisión más pausada:

* **Comparaciones desiguales entre modelos:** No todos los algoritmos competían con las mismas variables de entrada. En regresión, algunos modelos usaban únicamente la superficie sembrada, mientras que otros recibían además el departamento y la campaña. Si un modelo tiene acceso a más datos que otro, su ventaja proviene de la información extra y no de la calidad del algoritmo.
* **Dependencia de una sola partición de prueba:** Los resultados iniciales dependían de una única división aleatoria entre datos de entrenamiento y datos de evaluación. En conjuntos pequeños (como la prueba de clasificación con 84 casos), acertar o fallar un solo registro cambia el porcentaje final en más de un punto.
* **Falta de un modelo de referencia (baseline):** No se comparaban los modelos contra una regla simple, como predecir siempre el promedio. Sin esa base, no es posible saber cuánto valor real agrega un algoritmo complejo.
* **Evaluación en condiciones artificiales de clasificación:** La prueba se había balanceado al 50% verano y 50% invierno, pero en la realidad del agro boliviano el 90% del arroz se siembra en verano y solo el 10% en invierno. La precisión medida en un escenario 50/50 no se mantiene cuando se aplica al campo real.
* **Falta de diagnóstico sobre el error:** Se sabía que el error porcentual medio (MAPE) rondaba el 45% o 50%, pero no se conocía en qué tipo de comunidades o regiones fallaban más los modelos.

---

### 1.2 Análisis a fondo de la base de datos
Antes de analizar los algoritmos, es indispensable entender qué contiene la base de datos y cuáles son sus particularidades numéricas y agronómicas:

* **Tamaño y origen del conjunto:** La base original cuenta con 1,375 registros correspondientes a comunidades productoras de arroz en Bolivia. En la versión curada para regresión (V2) se retiraron únicamente 2 registros con superficies extremas superiores a las 3,000 hectáreas, quedando 1,373 observaciones representativas.
* **Fuerte asimetría en la superficie y la producción:** La gran mayoría de los productores son pequeños campesinos, mientras que un grupo reducido concentra casi todo el volumen cosechado.
  * La **mediana de superficie es de 17.0 hectáreas**, pero el promedio se eleva a **71.0 hectáreas** debido a parcelas que alcanzan hasta 2,425 hectáreas en la versión V2.
  * La **mediana de producción es de 500 quintales**, mientras que el promedio es de **2,938 quintales** (con un valor máximo de 172,903 quintales).
  * En términos prácticos: **el 13.5% de las comunidades (186 registros de 100 hectáreas o más) concentra el 79.6% de toda la producción de arroz registrada**.
* **La identidad matemática entre producción, superficie y rendimiento:** En la base original, el rendimiento en kilogramos por hectárea guarda una relación exacta con las otras dos variables:
  $$\text{Rendimiento (kg/ha)} = \frac{\text{Producción (qq)} \times 46}{\text{Superficie (ha)}}$$
  Esto ocurre porque en Bolivia un quintal de arroz equivale comercialmente a 46 kilogramos. Si se incluye el rendimiento para predecir la producción, el modelo no aprende un patrón agronómico real, sino que simplemente despeja una fórmula aritmética. Por esa razón, el rendimiento debió excluirse de la regresión.
* **Comportamiento del rendimiento según el tamaño:** En parcelas menores a 20 hectáreas, el rendimiento es bastante estable: el 80% de los casos se sitúa entre 965 y 2,718 kg/ha (una variación de 2.8 veces). En cambio, en parcelas grandes ($\ge 100\text{ Ha}$), el rendimiento oscila entre 308 y 3,049 kg/ha (casi 10 veces de diferencia), e incluso 25 de esas 186 comunidades rinden menos de 500 kg/ha debido a inundaciones, plagas o problemas de registro.
* **Distribución geográfica y estacional:** Santa Cruz representa 566 registros, Beni 311, Cochabamba 199, La Paz 179, Pando 109, Tarija 8 y Chuquisaca 1. El 80.7% del arroz de invierno (113 de 140 casos) se concentra exclusivamente en Santa Cruz, y el 18.6% restante (26 casos) en Beni. En departamentos como La Paz, Pando o Cochabamba, el cultivo invernal es prácticamente inexistente en este registro.
* **Comunidades con registros repetidos:** Existen 133 comunidades que figuran más de una vez en la base de datos (por ejemplo, reportadas tanto en la campaña de verano como en la de invierno). En estos casos, los rendimientos de un mismo lugar varían drásticamente según la época del año.

---

### 1.3 Regresión: comparación con mismas variables (script 19)
Para comparar de forma justa los ocho modelos de regresión, se aplicó una **validación cruzada repetida de 5 pliegues con 5 repeticiones (25 evaluaciones por modelo)**. Esto significa dividir los datos en cinco grupos, usar cuatro para entrenar y uno para evaluar, y repetir el proceso cinco veces barajando los datos. Así se obtiene un promedio y una desviación estándar ($\pm$) que mide qué tan estable es el resultado.

Se evaluaron dos situaciones:
* **Escenario A:** Predicción usando únicamente la **Superficie** sembrada.
* **Escenario B:** Predicción usando **Superficie + Departamento + Campaña**.

**Tabla 1.1. Regresión: Resultados promedio y variación en 25 evaluaciones**

| Modelo | $R^2$ en Quintales (Esc. A) | $R^2$ en Escala Log (Esc. A) | Error Absoluto MAE (qq) | Error Relativo MAPE (%) | $R^2$ en Quintales (Esc. B) |
|---|:---:|:---:|:---:|:---:|:---:|
| **Polinomial Ridge** | 0.7558 ± 0.1660 | 0.8344 ± 0.0286 | 1,223.7 ± 218.1 | 50.7 ± 10.3 | 0.7635 ± 0.1952 |
| **SVR (Kernel RBF)** | 0.7026 ± 0.2374 | 0.8331 ± 0.0337 | 1,225.1 ± 325.9 | 54.6 ± 12.1 | 0.7578 ± 0.2137 |
| **Árbol de Decisión** | 0.7076 ± 0.1228 | 0.8340 ± 0.0287 | 1,309.0 ± 326.3 | 48.7 ± 9.0 | 0.7098 ± 0.1236 |
| **LightGBM** | 0.6607 ± 0.1283 | 0.8375 ± 0.0286 | 1,396.1 ± 334.1 | 48.0 ± 8.3 | 0.6714 ± 0.1272 |
| **XGBoost** | 0.6506 ± 0.1428 | 0.8323 ± 0.0305 | 1,381.1 ± 349.2 | 48.7 ± 9.2 | 0.6533 ± 0.1559 |
| **CatBoost** | 0.6217 ± 0.1811 | 0.8384 ± 0.0288 | 1,350.0 ± 329.9 | 48.8 ± 9.4 | 0.6068 ± 0.2189 |
| **Regresión Lineal** | 0.6148 ± 0.0469 | 0.8250 ± 0.0275 | 1,488.6 ± 290.5 | 50.6 ± 8.2 | 0.6147 ± 0.0445 |
| **AdaBoost** | 0.5815 ± 0.1222 | 0.8231 ± 0.0214 | 1,558.7 ± 346.2 | 45.6 ± 6.5 | 0.5746 ± 0.1206 |
| **Baseline (Media)** | -0.0513 ± 0.0152 | -0.0031 ± 0.0052 | 2,670.9 ± 410.8 | 104.1 ± 5.9 | -0.0513 ± 0.0152 |

![Figura 7.1. Comparación equitativa de regresión (script 19)](../03_imagenes/41_comparacion_equitativa_reg_v4.png)

**Lectura clara de los resultados:**
1. **La superficie sí aporta información real:** El modelo base que solo predice el promedio obtiene un $R^2$ negativo (-0.05) y un error absoluto de 2,670 quintales. Todos los algoritmos entrenados reducen ese error a la mitad (entre 1,100 y 1,500 quintales).
2. **No hay un algoritmo claramente superior:** Aunque Polinomial Ridge aparece primero en la tabla, su variación ($\pm 0.166$) es más grande que la distancia que le saca a la Regresión Lineal o a los árboles. En estadística, si la diferencia entre dos modelos es menor que su margen de variación habitual, no se puede afirmar que uno sea mejor que otro.
3. **En escala relativa todos empatan:** Al mirar el $R^2$ logarítmico (que evalúa el ajuste porcentual sin dejarse distorsionar por las pocas fincas gigantescas), todos los modelos quedan entre **0.82 y 0.84**. Una simple regla lineal con dos números predice casi con la misma precisión relativa que una red de árboles complejos.
4. **Las variables de región y campaña casi no ayudan:** Al pasar del Escenario A al B, el $R^2$ cambia en cifras mínimas (entre -0.01 y +0.05). Una vez que conocemos el tamaño del terreno, saber en qué departamento está añade muy poco para calcular el volumen total de arroz.

---

### 1.4 Regresión: dónde y por qué se equivocan los modelos (script 20)
Para no quedarse solo con el promedio general, se evaluaron las predicciones fuera de muestra (registros predichos por modelos que nunca los vieron al entrenar). Se compararon la **Regresión Lineal** y **LightGBM** dividiendo las comunidades por tamaño de terreno:

**Tabla 1.2. Error de estimación según el tamaño del terreno**

| Rango de Superficie | N° Comunidades | Error MAE LightGBM (qq) | Error Relativo MAPE LightGBM (%) | Mediana del Error (%) | Tendencia / Sesgo (%) | Error Relativo Reg. Lineal (%) |
|---|:---:|:---:|:---:|:---:|:---:|:---:|
| **Menos de 5 ha** | 63 | 76.1 qq | 23.1% | 23.7% | +6.7% (Leve sobreestimación) | 37.3% |
| **De 5 a 20 ha** | 718 | 107.6 qq | 27.9% | 23.4% | +6.7% | 30.1% |
| **De 20 a 100 ha** | 406 | 526.3 qq | 47.6% | 30.1% | +16.8% | 48.7% |
| **100 ha o más** | 186 | 8,717.6 qq | 132.0% | 41.9% | +81.5% (Fuerte sobreestimación) | 139.1% |

![Figura 7.2. Diagnóstico de errores por grupo y escala (script 20)](../03_imagenes/44_errores_por_grupo_reg_v4.png)

**¿Qué nos enseña esta tabla sobre el comportamiento del modelo?**
* **El error relativo sube con el tamaño, no al revés:** Al inicio del proyecto se pensaba que las comunidades pequeñas causaban el alto porcentaje de error. La evidencia demostró lo contrario: en fincas pequeñas el error porcentual es bajo (23% a 28%), mientras que en fincas de más de 100 hectáreas el promedio de error se dispara a 132%.
* **La razón agronómica:** Como se vio en la base de datos, en terrenos grandes el rendimiento por hectárea es sumamente disparejo. Si un productor de 500 hectáreas tuvo un mal año y cosechó muy poco, el modelo matemático —que solo sabe que el terreno es inmenso— predice una cosecha enorme. Eso genera una sobreestimación que infla el porcentaje.
* **La diferencia entre promedio y mediana:** En las fincas de más de 100 hectáreas, el error promedio es 132%, pero la **mediana es del 41.9%**. Esto significa que a la mayoría de las fincas grandes se las predice con un 40% de error aproximado, pero unos pocos casos donde la cosecha fracasó por completo disparan el promedio aritmético.

---

### 1.5 Clasificación: comparación con mismas variables (script 17)
Se evaluaron nueve modelos y un clasificador aleatorio sobre la muestra balanceada (140 casos de verano y 140 de invierno), considerando como objetivo positivo detectar las parcelas de **invierno**.

**Tabla 1.3. Clasificación con variables numéricas + departamento (25 evaluaciones)**

| Modelo | Exactitud (Accuracy) | Detección de Invierno (Recall) | Confiabilidad de Alerta (Precision) | Balance F1-Score | Capacidad de Ordenar (ROC-AUC) |
|---|:---:|:---:|:---:|:---:|:---:|
| **XGBoost** | 76.4% ± 4.5% | 77.3% ± 5.4% | 76.3% ± 5.7% | **0.766 ± 0.043** | 0.855 ± 0.042 |
| **SVC (Soporte Vectorial)** | 75.1% ± 4.3% | 81.1% ± 7.8% | 73.0% ± 5.4% | **0.765 ± 0.041** | 0.846 ± 0.034 |
| **Naive Bayes Gaussiano** | 69.3% ± 3.2% | 99.3% ± 1.4% | 62.2% ± 2.5% | **0.764 ± 0.019** | 0.821 ± 0.057 |
| **Regresión Logística** | 74.6% ± 4.0% | 82.1% ± 4.9% | 71.6% ± 4.5% | **0.764 ± 0.035** | 0.806 ± 0.038 |
| **CatBoost** | 76.1% ± 4.3% | 77.0% ± 5.8% | 76.1% ± 5.4% | **0.763 ± 0.041** | 0.854 ± 0.042 |
| **AdaBoost** | 74.1% ± 3.8% | 81.4% ± 5.1% | 71.3% ± 4.4% | **0.759 ± 0.033** | 0.790 ± 0.040 |
| **LightGBM** | 75.5% ± 5.5% | 76.6% ± 8.0% | 75.2% ± 5.5% | **0.757 ± 0.056** | 0.851 ± 0.045 |
| **KNN (Vecinos Cercanos)** | 75.0% ± 5.1% | 77.7% ± 8.1% | 74.0% ± 5.7% | **0.756 ± 0.053** | 0.829 ± 0.047 |
| **Árbol de Decisión** | 74.5% ± 5.3% | 74.1% ± 10.7% | 75.6% ± 7.7% | **0.742 ± 0.060** | 0.820 ± 0.050 |
| **Azar (Baseline)** | 51.6% ± 7.0% | 51.6% ± 7.0% | 51.6% ± 7.0% | 0.516 ± 0.070 | 0.516 ± 0.070 |

![Figura 7.3. Comparación equitativa de clasificación (script 17)](../03_imagenes/42_comparacion_equitativa_clf_v4.png)

**Lectura clara de los resultados:**
* **Empate técnico general:** Con las mismas variables, **nueve modelos quedan separados por apenas dos centésimas de F1-Score (entre 0.74 y 0.76)**. La supuesta superioridad que antes mostraban CatBoost o XGBoost venía de que en versiones previas ellos tenían acceso al departamento mientras los otros modelos no.
* **El departamento es la variable decisiva:** Cuando a los modelos se les quita el departamento y solo usan números (superficie, producción y rendimiento), el F1 cae a 0.68 o 0.70. Al incluir el departamento, todos suben a 0.76. Esto no es magia del algoritmo: como vimos en la base de datos, más del 80% del arroz de invierno está en Santa Cruz. El modelo simplemente aprende que si una comunidad no es de Santa Cruz o Beni, la probabilidad de invierno es casi cero.
* **El caso engañoso de Naive Bayes:** Naive Bayes muestra un F1 alto (0.764) y un Recall del 99.3%, pero su exactitud global es la más baja (69.3%) y su precisión cae a 62.2%. Lo que hace este modelo es marcar casi todo como invierno; no se le escapa casi ningún caso de invierno, pero genera muchísimas falsas alarmas con comunidades de verano.

---

### 1.6 Clasificación en la distribución real 90/10 (script 18)
Hasta aquí los clasificadores se evaluaron en un ambiente controlado 50/50. Pero en el campo boliviano, solo 1 de cada 10 parcelas es de invierno. El script 18 evaluó qué pasa cuando se prueba el modelo con la proporción real (10.2% invierno vs 89.8% verano) usando tres estrategias de entrenamiento:

**Tabla 1.4. Desempeño bajo la proporción real del agro boliviano (90/10)**

| Modelo | Estrategia de Entrenamiento | Confiabilidad Real (Precisión) | Detección Real (Recall) | F1-Score Real | Calidad de Ordenamiento (ROC-AUC) |
|---|---|:---:|:---:|:---:|:---:|
| **CatBoost** | Pesos de clase (Balanced) | 29.6% ± 2.9% | 71.4% ± 8.7% | 0.418 ± 0.041 | 0.842 ± 0.025 |
| **XGBoost** | Pesos de clase (Balanced) | 29.9% ± 2.2% | 71.2% ± 8.3% | 0.421 ± 0.034 | 0.837 ± 0.029 |
| **Regresión Logística** | Pesos de clase (Balanced) | 20.0% ± 1.7% | 82.4% ± 5.8% | 0.321 ± 0.024 | 0.795 ± 0.029 |
| **CatBoost** | Sin balancear (Natural) | 37.7% ± 10.9% | 11.0% ± 7.4% | 0.159 ± 0.088 | 0.847 ± 0.027 |
| **XGBoost** | Sin balancear (Natural) | 45.6% ± 27.8% | 9.3% ± 5.3% | 0.146 ± 0.077 | 0.837 ± 0.029 |

![Figura 7.4. Clasificación con la distribución real 90/10 (script 18)](../03_imagenes/43_distribucion_real_clf_v4.png)

**¿Por qué la precisión se cae del 76% al 30%?**
* **El efecto matemático del desbalance:** El modelo no se ha vuelto deficiente; su ROC-AUC sigue siendo muy bueno (0.84), lo que indica que sabe ordenar las parcelas de mayor a menor probabilidad de invierno.
* **El problema de las falsas alarmas:** Imaginemos 1,000 parcelas reales del país: unas 100 serán de invierno y 900 de verano. El modelo detectará unas 71 de las 100 de invierno (buen Recall). Pero como comete un pequeño error en las de verano (un 18% de falsas alarmas), clasificará por error a unas 160 parcelas de verano como si fueran de invierno.
* **En total habrá unas 230 alarmas, de las cuales solo 71 son reales:** Por eso la precisión es de aproximadamente el **30%**. De cada diez veces que el modelo diga «este arroz es de invierno», solo tres serán verdad y siete serán falsas alarmas. Esto es crucial saberlo antes de llevar un sistema a la práctica.

---

### 1.7 Efecto de los casos extremos en el conjunto original
Para comprobar la importancia de limpiar los datos, se revisaron los experimentos que corrían sobre los 1,375 registros originales sin filtrar. En ellos figuraba una comunidad con **8,545 hectáreas y más de 804,000 quintales**.
* Cuando esa sola comunidad caía dentro del grupo de prueba, el $R^2$ de los modelos caía a **0.36**.
* Cuando esa comunidad caía en el grupo de entrenamiento, el $R^2$ subía a **0.95**.

Un solo dato extremo hacía que el mismo algoritmo pareciera excelente o mediocre según la suerte del reparto aleatorio. Esto demuestra que **el filtrado de datos atípicos y la validación cruzada no son detalles menores, sino el único camino para obtener mediciones confiables**.

---

## 2. Discusión de los resultados

* **El techo natural de la predicción de cosechas:** Con los datos disponibles, ningún algoritmo podrá predecir la producción con un error cercano a cero. La producción real depende del rendimiento por hectárea, y el rendimiento varía ampliamente por factores que la base de datos no tiene registrados (riego, lluvias, plagas o fertilizantes). Como el modelo solo conoce las hectáreas sembradas, el límite de precisión alcanzado ($R^2$ relativo de 0.83) representa prácticamente todo lo que se puede extraer de estas variables.
* **La sencillez frente a la complejidad:** En problemas donde una sola variable domina el fenómeno (como la superficie sobre la producción), los modelos lineales bien calibrados rinden casi a la par de los ensambles más avanzados. La complejidad algorítmica solo aporta ventajas cuando existen múltiples variables interactuando entre sí.
* **Ninguna métrica cuenta toda la verdad:**
  * El $R^2$ en quintales se deja engañar por las fincas gigantes.
  * La exactitud (Accuracy) en clasificación se deja engañar por las clases desbalanceadas (decir siempre «verano» daría 90% de aciertos sin haber aprendido nada).
  * Por eso siempre deben mirarse varias métricas a la vez: el error absoluto (MAE), el relativo (MAPE), la sensibilidad (Recall) y la precisión real.

---

## 3. Selección de modelos para la aplicación web

A partir de los resultados comparativos y considerando la facilidad de uso e implementación, se definen las siguientes recomendaciones para el sistema web:

1. **Para estimar la producción (Regresión):**
   * **Opción principal (Directa en el navegador, sin servidor): Regresión Lineal V3.**
     Alcanza un $R^2$ relativo de 0.825 y es la más estable entre todas las particiones. Puede programarse con una sola línea de código en JavaScript dentro de la página:
     $$\text{Producción} = \exp\Big(3.5059 + 0.9667 \times \ln(\text{Superficie} + 1)\Big) - 1$$
     *Advertencia al usuario:* Se debe informar claramente en pantalla que se trata de una estimación orientativa, con un margen de variación promedio de alrededor del 45%.
   * **Opción alternativa (API en Python): LightGBM o Polinomial Ridge.**
     Ofrecen errores absolutos ligeramente menores en fincas medianas si se cuenta con un servidor backend activo.

2. **Para clasificar la campaña agrícola (Clasificación):**
   * **Modelo recomendado: CatBoost o XGBoost entrenados con pesos de clase.**
     Ambos logran un ROC-AUC de 0.84 y capturan más del 70% de las parcelas de invierno.
     *Advertencia operativa:* Dado que en la distribución real 90/10 la precisión se sitúa en torno al 30%, el sistema no debe tomar decisiones automáticas, sino emitir una sugerencia de apoyo para revisión humana.

---

## 4. Conclusiones

1. Los algoritmos de regresión y clasificación evaluados mostraron rendimientos estadísticamente equivalentes cuando se los comparó bajo las mismas variables y mediante validación cruzada. Ningún algoritmo superó de forma definitiva a los demás.
2. La superficie cultivada es la variable determinante para calcular la producción de arroz. Agregar el departamento o la campaña no mejora la estimación de quintales de forma medible.
3. El error relativo no proviene de los pequeños productores, sino de las parcelas grandes ($\ge 100\text{ Ha}$), donde el rendimiento agronómico es altamente impredecible a partir únicamente del área sembrada.
4. En clasificación, la aparente ventaja de ciertos modelos se debía a que conocían el departamento. Al estar el 80.7% del invierno concentrado en Santa Cruz, el departamento actúa como un clasificador geográfico indirecto.
5. Las pruebas con la proporción real del país (90/10) demostraron que la precisión práctica para detectar invierno ronda el 30%. Evaluar únicamente en conjuntos balanceados de laboratorio genera expectativas que no se cumplen en el mundo real.

---

## 5. Recomendaciones y pasos a seguir para proyectos de aprendizaje supervisado

A partir de los tropiezos identificados y corregidos a lo largo de este proyecto, se proponen los siguientes pasos para quien vaya a analizar su primera base de datos en aprendizaje supervisado:

1. **Revisar a fondo la distribución antes de tocar cualquier modelo:**
   No te fíes del promedio. Calcula siempre la mediana y los percentiles. Si el promedio de tu variable es mucho mayor que la mediana (como nos pasó aquí con 71 ha frente a 17 ha), significa que tienes datos concentrados en pocas filas gigantescas. En esos casos, transformar con logaritmo ($\ln(1+x)$) desde el primer día te ahorrará distorsiones severas.
2. **Cazar activamente la fuga de datos (data leakage):**
   Pregúntate siempre: *¿esta columna se conoce antes de que ocurra lo que quiero predecir?* Si dos columnas están amarradas por una fórmula física o contable (como Superficie, Rendimiento y Producción), usar una para predecir la otra no es ciencia de datos; es engañarse a uno mismo. Revisa las fórmulas matemáticas implícitas entre tus columnas.
3. **Poner siempre un modelo base simple (Baseline):**
   Antes de entrenar redes neuronales o Gradient Boosting, programa una regla sencilla: ¿qué pasa si siempre predigo el promedio? ¿Qué pasa si siempre predigo la clase mayoritaria? Si tu algoritmo avanzado apenas empata con esa regla simple, el modelo no está aprendiendo nada útil.
4. **No conformarse con una sola partición de prueba:**
   Separar los datos una sola vez con `train_test_split` es peligroso en conjuntos pequeños o con datos atípicos. Un solo registro raro que caiga en la prueba puede hacer que tu modelo parezca un fracaso o un éxito total. Usa validación cruzada repetida y reporta siempre el promedio junto a su desviación estándar.
5. **Comprobar la regla de solapamiento:**
   Si el modelo A tiene un $R^2$ de 0.70 con una variación de $\pm 0.15$, y el modelo B tiene 0.65 con $\pm 0.10$, no digas que el modelo A es mejor. Sus márgenes de error se tocan. Aprende a aceptar que a veces varios algoritmos son técnicamente equivalentes y elige el más simple de mantener.
6. **Cuidado con el desbalance de clases:**
   Si tu evento de interés ocurre solo el 10% de las veces, la métrica de exactitud (Accuracy) no sirve. Si balanceas los datos para ayudar al modelo a aprender, recuerda volver a probarlo sobre los datos desbalanceados originales. Es ahí donde verás la verdadera precisión y la cantidad real de falsas alarmas que el usuario final tendrá que soportar.
7. **Buscar el balance entre simplicidad y costo de despliegue:**
   Si una ecuación de dos parámetros te da el 98% del rendimiento de un ensamble que pesa 50 megabytes y requiere librerías pesadas en un servidor, quédate con la ecuación simple. En ingeniería de datos, el mejor modelo no es el más sofisticado, sino el que resuelve la necesidad del usuario con la menor cantidad de piezas móviles.